# Exploratory analysis: static geometry exploration

 This notebook is outside the main paper workflow. Existing results and metric variants are historical/exploratory, not the authoritative paper results. Original analysis cells and outputs are retained. Some legacy sections require selective execution; this is not a verified clean-run pipeline.

The main workflow is in `notebooks/paper/`. This notebook may write legacy exports under `analysis_exports/`; the paper pipeline uses its own `outputs/paper/` directory.

Plot defaults now come from `plot_style.py`. Prior static image outputs were cleared; rerun plot cells after their prerequisites to see the shared style. Specialized heatmap scales and animations retain their own semantic encodings.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data/wired").is_dir() and (p / "paper").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from this repository or a notebook directory inside it.")
os.chdir(PROJECT_ROOT)  # Preserve project-relative paths when launched from a subdirectory.
print("Project root:", PROJECT_ROOT)

# Shared project plotting conventions.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from plot_style import (
    apply_style, expertise_palette, EXPERTISE_COLORS, LEVEL_LABELS,
    METRIC_LABELS, MODEL_COLORS, MODEL_MARKERS, PRIMARY, NEUTRAL,
    plot_metric_trajectories, save_figure,
)
apply_style()


# Static conversational geometry

Conversation-level semantic geometry, matched visualizations, multilevel models, speaker-centroid separation, and radial outlier structure.

This notebook was separated from `big_analysis_centroid_fixed(1).ipynb`. Stale outputs were removed so results are regenerated from the current data. Run cells from top to bottom with the working directory set to the project root containing `data/wired`.

Load csvs. 1 csv- one conversation. each row, one utterance. some consecutive speaker rows.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path.cwd()
WIRED_DIR = PROJECT_ROOT / "data" / "wired"

csv_paths = sorted(WIRED_DIR.glob("wired_*/*.csv"))

print(f"Found {len(csv_paths)} CSV files")
csv_paths[:10]

Found 105 CSV files


[PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_12.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_13.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_14.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_15.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_16.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_12.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_13.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_14.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_15.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_

In [2]:
def parse_wired_path(path):
    """
    Example:
    data/wired/wired_astro/wired_astro_12.csv
    """
    folder_name = path.parent.name
    file_stem = path.stem

    video_id = re.sub(r"^wired_", "", folder_name)

    match = re.search(r"_(\d+)$", file_stem)
    file_number = int(match.group(1)) if match else None

    return {
        "dataset": "wired",
        "video_id": video_id,
        "file_number": file_number,
        "conversation_id": file_stem,
        "source_file": str(path.relative_to(PROJECT_ROOT))
    }

In [3]:
raw_frames = []

for path in csv_paths:
    print("Reading:", path)

    try:
        df = pd.read_csv(path)
    except pd.errors.ParserError as e:
        print("\n❌ BROKEN FILE:", path)
        print("ERROR:", e)
        raise

    metadata = parse_wired_path(path)

    df["raw_row_id"] = np.arange(len(df))

    for column, value in metadata.items():
        df[column] = value

    raw_frames.append(df)

Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_12.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_13.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_14.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_15.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_16.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_12.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_13.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_14.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_15.csv
Reading: /Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_16.csv
Reading: /Users/yume/PycharmProjects/horizons_2/da

In [4]:
raw_frames = []

for path in csv_paths:
    df = pd.read_csv(path)
    metadata = parse_wired_path(path)

    # Preserve original within-file row order
    df["raw_row_id"] = np.arange(len(df))

    for column, value in metadata.items():
        df[column] = value

    raw_frames.append(df)

wired_raw = pd.concat(
    raw_frames,
    ignore_index=True,
    sort=False
)

wired_raw.shape

(6381, 10)

table with all utterances, all conversations

In [5]:
wired_raw.head()

,Sequence,Speaker,Utterance,Notes,raw_row_id,dataset,video_id,file_number,conversation_id,source_file
0,1,Speaker 2 (child),Hi.,NaN,0,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
1,2,Speaker 1 (Janna Levin),"Hi, welcome.",NaN,1,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
2,3,Speaker 1 (Janna Levin),Tell me your name.,NaN,2,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
3,4,Speaker 2 (child),Jude.,NaN,3,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
4,5,Speaker 1 (Janna Levin),I wanted to ask you if you have ever heard of ...,NaN,4,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv


In [6]:
#rename columns, minimally clean text
wired_raw = wired_raw.rename(columns={
    "Speaker": "speaker_raw",
    "Utterance": "text"
})
wired_raw["text"] = (
    wired_raw["text"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

wired_raw = wired_raw[
    wired_raw["text"].notna() &
    wired_raw["text"].ne("")
].copy()

In [7]:
level_map = {
    12: (0, "child"),
    13: (1, "teenager"),
    14: (2, "undergraduate"),
    15: (3, "graduate"),
    16: (4, "expert")
}

In [8]:
wired_raw["level"] = wired_raw["file_number"].map(
    lambda x: level_map[x][0]
)

wired_raw["level_label"] = wired_raw["file_number"].map(
    lambda x: level_map[x][1]
)

In [9]:
#order levels
level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert"
]

wired_raw["level_label"] = pd.Categorical(
    wired_raw["level_label"],
    categories=level_order,
    ordered=True
)

In [10]:
'''
speaker_mapping = {
    #astro
     "Speaker 1 (Janna Levin)": "A",
     "Speaker 2 (child)": "B",
     "Speaker 3 (teen)": "B",
    "Speaker 4 (college student)": "B",
    "Speaker 5 (grad student)": "B",
    "Speaker 6 (expert)": "B",
    #blockchain
    "Speaker 1 (Bettina Warburg)": "A",
    "Speaker 2 (Child)": "B",
    "Speaker 3 (Teen - Ian)": "B",
    "Speaker 4 (College Student)": "B",
    "Speaker 5 (Grad Student)": "B",
    "Speaker 6 (Expert)": "B",
    #crispr
    "Speaker 1 (Neville Sanjana)": "A",
    "Speaker 2 (Tegan - Child)": "B",
    "Speaker 3 (Bella - Teenager)": "B",
    "Speaker 4 (Christopher - Student)": "B",
    "Speaker 5 (Lauren Schiff)": "B",
    "Speaker 6 (Matthew Canver)": "B",
    #dimension
    "Speaker 1 (Sean Carroll PhD)": "A",
    #fractal
    "Speaker 1 (Keenan Crane Phd)": "A",
    #gravity
    "Speaker 1 (Janna Levin)": "A",
    "Speaker 2 (Bonét Sofía Kanayet)": "B",
    "Speaker 3 (Maria Teresa Furtado)": "B",
    "Speaker 4 (Lisa Chan)": "B",
    "Speaker 5 (Will Gyory)": "B",
    "Speaker 6 (Matthew Kleban)": "B",
    #hacking
    "Speaker 1 (Samy Kamkar)": "A",
    #infinity
    "Speaker 1 (Emily Riehl)": "A",
    #internet
    "Speaker 1 (Jim Kurose)": "A",
    #laser
    "Speaker 1 (Donna Strickland)": "A",
    "Speaker 2 (Harmoni - Child)": "B",
    "Speaker 3 (Eli Kaplan - Teenager)": "B",
    "Speaker 4 (Caitlin - Student)": "B",
    "Speaker 5 (Aditya - Grad Student)": "B",
    "Speaker 6 (Mike Campbell)": "B",
    #machine
    "Speaker 1 (Hilary Mason)": "A",
    #memory
    "Speaker 1 (Daphna Shohamy)": "A",
    #moravec
    "Speaker 1 (Chelsea Finn)": "A",
    #nano
    "Speaker 1 (Dr. George S. Tulevski)": "A",
    #neuro
    "Speaker 1 (Dr. Bobby Kasthuri)": "A",
    "Speaker 2 (child - Daniel)": "B",
    "Speaker 3 (Jabez Griggs)": "B",
    "Speaker 4 (Elena Dowling)": "B",
    "Speaker 5 (Mala Ananth)": "B",
    "Speaker 6 (Russell Hanson)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",


}
'''
is_expert = wired_raw["speaker_raw"].str.contains(
    r"\bSpeaker\s*1\b",
    case=False,
    na=False
)

wired_raw["speaker"] = np.where(is_expert, "A", "B")
wired_raw["speaker_role"] = np.where(
    is_expert,
    "expert",
    "partner"
)




wired_raw["speaker_role"] = wired_raw["speaker"].map({
    "A": "expert",
    "B": "partner"
})

# Utterance embeddings

## utterances table -> embed utterances

In [11]:
import re
import numpy as np
import pandas as pd


def count_words(text):
    return len(
        re.findall(
            r"\b[\w'-]+\b",
            str(text),
        )
    )


def create_utterances_table(wired_raw):
    """
    Create one canonical row per original transcript utterance.

    The function does not concatenate adjacent utterances.
    """

    utterances = (
        wired_raw
        .copy()
        .reset_index(drop=False)
        .rename(columns={"index": "_original_dataframe_row"})
    )

    # Accommodate either the original CSV column names or the
    # standardized names already present in wired_raw.
    rename_map = {}

    aliases = {
        "Sequence": "sequence",
        "Utterance": "text",
        "Speaker": "speaker_raw",
        "Notes": "notes",
    }

    for old_name, new_name in aliases.items():
        if (
            old_name in utterances.columns
            and new_name not in utterances.columns
        ):
            rename_map[old_name] = new_name

    utterances = utterances.rename(columns=rename_map)

    # If canonical speaker has not already been created, use
    # the original speaker label.
    if "speaker" not in utterances.columns:
        utterances["speaker"] = utterances["speaker_raw"]

    # Stable source order.
    if "sequence" not in utterances.columns:
        utterances["sequence"] = (
            utterances
            .groupby(
                ["dataset", "conversation_id"],
                sort=False,
            )
            .cumcount()
            .add(1)
        )

    utterances["_sequence_numeric"] = pd.to_numeric(
        utterances["sequence"],
        errors="coerce",
    )

    utterances["_sequence_numeric"] = (
        utterances["_sequence_numeric"]
        .fillna(utterances["_original_dataframe_row"])
    )

    conversation_keys = [
        "dataset",
        "conversation_id",
    ]

    sort_columns = [
        *conversation_keys,
        "_sequence_numeric",
        "_original_dataframe_row",
    ]

    utterances = (
        utterances
        .sort_values(
            sort_columns,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # Clean text without altering its linguistic contents.
    utterances["text"] = (
        utterances["text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    if utterances["text"].eq("").any():
        empty_rows = utterances.loc[
            utterances["text"].eq(""),
            [
                *conversation_keys,
                "sequence",
            ],
        ]

        raise ValueError(
            "Empty utterance text found:\n"
            + empty_rows.to_string(index=False)
        )

    # Reconstruct turn membership without concatenating anything.
    # A new turn begins whenever the speaker changes.
    speaker_changed = (
        utterances
        .groupby(
            conversation_keys,
            sort=False,
        )["speaker"]
        .transform(
            lambda speakers:
                speakers.ne(speakers.shift())
        )
    )

    utterances["turn_id"] = (
        speaker_changed
        .astype(int)
        .groupby(
            [
                utterances[column]
                for column in conversation_keys
            ]
        )
        .cumsum()
    )

    # Position of the utterance inside its speaker turn.
    utterances["utterance_in_turn"] = (
        utterances
        .groupby(
            [
                *conversation_keys,
                "turn_id",
            ],
            sort=False,
        )
        .cumcount()
        .add(1)
    )

    # Position within the whole conversation.
    utterances["utterance_number"] = (
        utterances
        .groupby(
            conversation_keys,
            sort=False,
        )
        .cumcount()
        .add(1)
    )

    utterances["utterance_id"] = (
        utterances["dataset"].astype(str)
        + "::"
        + utterances["conversation_id"].astype(str)
        + "::utterance_"
        + utterances["utterance_number"]
            .astype(str)
            .str.zfill(3)
    )

    utterances["n_words"] = (
        utterances["text"]
        .map(count_words)
        .astype(int)
    )

    # Keep all utterances in the canonical table. This flag only
    # determines the primary geometry subset later.
    utterances["include_geometry_primary"] = (
        utterances["n_words"] >= 5
    )

    # This index will correspond exactly to rows of E_utterances.
    utterances["embedding_idx"] = np.arange(
        len(utterances),
        dtype=int,
    )

    preferred_columns = [
        "dataset",
        "video_id",
        "conversation_id",
        "file_number",
        "level",
        "level_label",
        "utterance_id",
        "utterance_number",
        "turn_id",
        "utterance_in_turn",
        "sequence",
        "normalized_time",
        "speaker",
        "speaker_raw",
        "speaker_role",
        "text",
        "n_words",
        "include_geometry_primary",
        "embedding_idx",
        "notes",
        "source_file",
        "_original_dataframe_row",
    ]

    # Retain only columns that actually exist in this dataset.
    output_columns = [
        column
        for column in preferred_columns
        if column in utterances.columns
    ]

    return utterances[output_columns].copy()


utterances = create_utterances_table(wired_raw)

utterances.head()

,dataset,video_id,conversation_id,file_number,level,level_label,utterance_id,utterance_number,turn_id,utterance_in_turn,...,speaker,speaker_raw,speaker_role,text,n_words,include_geometry_primary,embedding_idx,notes,source_file,_original_dataframe_row
0,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_001,1,1,1,...,B,Speaker 2 (Genesis - child),partner,what's this,2,False,0,NaN,data/wired/wired_talia/wired_Talia_12.csv,5383
1,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_002,2,2,1,...,A,Speaker 1 (Talia Gershon),expert,yeah what do you think that is,7,True,1,NaN,data/wired/wired_talia/wired_Talia_12.csv,5384
2,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_003,3,3,1,...,B,Speaker 2 (Genesis - child),partner,fancy chandelier,2,False,2,NaN,data/wired/wired_talia/wired_Talia_12.csv,5385
3,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_004,4,4,1,...,A,Speaker 1 (Talia Gershon),expert,i think so too we jokingly call it the chandelier,10,True,3,NaN,data/wired/wired_talia/wired_Talia_12.csv,5386
4,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_005,5,4,2,...,A,Speaker 1 (Talia Gershon),expert,that's real gold you know,5,True,4,NaN,data/wired/wired_talia/wired_Talia_12.csv,5387


In [12]:
from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())
print("Maximum sequence length:", embedding_model.max_seq_length)

/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding dimension: 384
Maximum sequence length: 256


In [13]:
# Prepare and validate utterance texts
utterance_texts = utterances["text"].astype(str).tolist()

utterances["n_model_tokens"] = [
    len(
        embedding_model.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )
    for text in utterance_texts
]

utterances["exceeds_model_limit"] = (
    utterances["n_model_tokens"]
    > embedding_model.max_seq_length
)

if utterances["exceeds_model_limit"].any():
    raise ValueError(
        "At least one utterance exceeds the model's token limit."
    )

# Generate one normalized embedding per utterance
E_utterances = embedding_model.encode(
    utterance_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

E_utterances = np.asarray(E_utterances, dtype=np.float32)

assert E_utterances.shape[0] == len(utterances)
assert np.isfinite(E_utterances).all()
assert np.allclose(
    np.linalg.norm(E_utterances, axis=1),
    1.0,
    atol=1e-5,
)

print("E_utterances shape:", E_utterances.shape)

Batches:   0%|          | 0/200 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Batches: 100%|██████████| 200/200 [00:03<00:00, 53.29it/s]


E_utterances shape: (6381, 384)


In [14]:
geometry_utterances = utterances.copy()

print("All utterances:", len(utterances))
print(
    "Primary geometry utterances:",
    len(geometry_utterances),
)
print(
    "Excluded short utterances:",
    len(utterances) - len(geometry_utterances),
)

All utterances: 6381
Primary geometry utterances: 6381
Excluded short utterances: 0


## analysis - look at distribution/sturcture of information

## conversation level geometry

In [15]:
import numpy as np
import pandas as pd


def _sample_sd(values):
    """Sample standard deviation, or NaN if only one value exists."""
    values = np.asarray(values)

    if values.size < 2:
        return np.nan

    return float(np.std(values, ddof=1))


def calculate_conversation_geometry(embeddings, n_words=None, n_turns=None, k=5):
    """
    Calculate the complete geometric metric set for one conversation.

    Parameters
    ----------
    embeddings : array-like, shape (n_utterances, embedding_dimension)
        One embedding per utterance.

    n_words : int, optional
        Total number of words in the conversation.

    n_turns : int, optional
        Total number of conversational turns.

    k : int, default=5
        Neighbor rank used for the k-nearest-neighbor statistic.

    Returns
    -------
    scalar_metrics : dict
        Conversation-level scalar metrics suitable for a DataFrame.

    spectrum : dict
        Eigenvalue arrays retained separately for spectral plots.
    """

    X = np.asarray(embeddings, dtype=np.float64)

    if X.ndim != 2:
        raise ValueError(
            "embeddings must have shape "
            "(n_utterances, embedding_dimension)"
        )

    if not np.isfinite(X).all():
        raise ValueError("Embeddings contain NaN or infinite values.")

    n, d = X.shape

    if n < 2:
        raise ValueError(
            "At least two utterances are required."
        )

    if not isinstance(k, (int, np.integer)) or k < 1:
        raise ValueError("k must be a positive integer.")

    # ------------------------------------------------------------
    # 1. Normalize embeddings
    # ------------------------------------------------------------

    norms = np.linalg.norm(X, axis=1, keepdims=True)

    if np.any(norms == 0):
        raise ValueError("At least one embedding has zero norm.")

    X_unit = X / norms

    # ------------------------------------------------------------
    # 2. Pairwise cosine similarity and distance
    # ------------------------------------------------------------

    cosine_similarity_matrix = np.clip(
        X_unit @ X_unit.T,
        -1.0,
        1.0,
    )

    cosine_distance_matrix = 1.0 - cosine_similarity_matrix

    # Set the diagonal to exactly zero for neighbor calculations.
    np.fill_diagonal(cosine_distance_matrix, 0.0)

    upper_triangle = np.triu_indices(n, k=1)

    pairwise_similarities = cosine_similarity_matrix[
        upper_triangle
    ]

    pairwise_distances = cosine_distance_matrix[
        upper_triangle
    ]

    n_pairs = len(pairwise_distances)

    sim_q10, sim_median, sim_q90 = np.quantile(
        pairwise_similarities,
        [0.10, 0.50, 0.90],
    )

    dist_q10, dist_median, dist_q90 = np.quantile(
        pairwise_distances,
        [0.10, 0.50, 0.90],
    )

    # ------------------------------------------------------------
    # 3. Centroid concentration
    # ------------------------------------------------------------

    centroid = X_unit.mean(axis=0)

    centroid_resultant_length = float(
        np.linalg.norm(centroid)
    )

    centroid_dispersion_cosine = float(
        1.0 - centroid_resultant_length
    )

    # ------------------------------------------------------------
    # 4. Radial structure
    # ------------------------------------------------------------

    centroid_tolerance = (
        100 * np.finfo(np.float64).eps
    )

    if centroid_resultant_length > centroid_tolerance:
        centroid_direction = (
            centroid / centroid_resultant_length
        )

        radial_distances = np.clip(
            1.0 - X_unit @ centroid_direction,
            0.0,
            2.0,
        )

        radial_distance_mean = float(
            np.mean(radial_distances)
        )

        radial_distance_median = float(
            np.median(radial_distances)
        )

        radial_distance_q90 = float(
            np.quantile(radial_distances, 0.90)
        )

    else:
        # A zero centroid has no defined direction.
        radial_distances = np.full(n, np.nan)
        radial_distance_mean = np.nan
        radial_distance_median = np.nan
        radial_distance_q90 = np.nan

    # ------------------------------------------------------------
    # 5. Local-neighborhood structure
    # ------------------------------------------------------------

    neighbor_distance_matrix = (
        cosine_distance_matrix.copy()
    )

    # Prevent each utterance from selecting itself.
    np.fill_diagonal(
        neighbor_distance_matrix,
        np.inf,
    )

    nearest_neighbor_distances = np.min(
        neighbor_distance_matrix,
        axis=1,
    )

    k_used = min(k, n - 1)

    kth_neighbor_distances = np.partition(
        neighbor_distance_matrix,
        kth=k_used - 1,
        axis=1,
    )[:, k_used - 1]

    # ------------------------------------------------------------
    # 6. Centered covariance spectrum
    # ------------------------------------------------------------

    X_centered = X_unit - centroid

    singular_values = np.linalg.svd(
        X_centered,
        compute_uv=False,
        full_matrices=False,
    )

    # Covariance eigenvalues:
    # lambda_j = s_j^2 / (n - 1)
    eigenvalues = (
        singular_values**2 / (n - 1)
    )

    total_variance = float(
        eigenvalues.sum()
    )

    maximum_available_rank = min(
        n - 1,
        d,
    )

    leading_singular_value = (
        float(singular_values[0])
        if singular_values.size
        else 0.0
    )

    sv_tolerance = (
        max(n, d)
        * np.finfo(np.float64).eps
        * leading_singular_value
    )

    active = singular_values > sv_tolerance

    # ------------------------------------------------------------
    # 7. Spectral metrics
    # ------------------------------------------------------------

    if np.isclose(
        total_variance,
        0.0,
        atol=1e-15,
        rtol=0.0,
    ):
        numerical_rank = 0
        normalized_eigenvalues = np.array([])

        spectral_entropy = np.nan
        spectral_entropy_normalized = np.nan
        spectral_effective_rank = np.nan
        participation_ratio = np.nan
        pc1_variance_share = np.nan

    else:
        numerical_rank = int(active.sum())

        active_eigenvalues = eigenvalues[active]

        normalized_eigenvalues = (
            active_eigenvalues
            / active_eigenvalues.sum()
        )

        # H = -sum(p_j log p_j)
        spectral_entropy = float(
            -np.sum(
                normalized_eigenvalues
                * np.log(normalized_eigenvalues)
            )
        )

        # Effective rank = exp(H)
        spectral_effective_rank = float(
            np.exp(spectral_entropy)
        )

        # PR = 1 / sum(p_j^2)
        participation_ratio = float(
            1.0
            / np.sum(normalized_eigenvalues**2)
        )

        # Proportion of variance explained by PC1
        pc1_variance_share = float(
            eigenvalues[0] / total_variance
        )

        if maximum_available_rank > 1:
            spectral_entropy_normalized = float(
                spectral_entropy
                / np.log(maximum_available_rank)
            )
        else:
            spectral_entropy_normalized = np.nan

    # ------------------------------------------------------------
    # 8. Collect scalar results
    # ------------------------------------------------------------

    scalar_metrics = {
        # Bookkeeping
        "n_utterances": int(n),
        "n_pairs": int(n_pairs),
        "n_words": (
            int(n_words)
            if n_words is not None
            else np.nan
        ),
        "n_turns": (
            int(n_turns)
            if n_turns is not None
            else np.nan
        ),
        "embedding_dimension": int(d),
        "k_used": int(k_used),

        # Pairwise cosine similarity
        "pairwise_cosine_similarity_mean": float(
            np.mean(pairwise_similarities)
        ),
        "pairwise_cosine_similarity_sd": _sample_sd(
            pairwise_similarities
        ),
        "pairwise_cosine_similarity_q10": float(
            sim_q10
        ),
        "pairwise_cosine_similarity_median": float(
            sim_median
        ),
        "pairwise_cosine_similarity_q90": float(
            sim_q90
        ),

        # Centroid concentration
        "centroid_resultant_length": (
            centroid_resultant_length
        ),
        "centroid_dispersion_cosine": (
            centroid_dispersion_cosine
        ),

        # Radial structure
        # The mean is retained mainly as an identity check.
        "radial_distance_mean_check": (
            radial_distance_mean
        ),
        "radial_distance_median": (
            radial_distance_median
        ),
        "radial_distance_q90": (
            radial_distance_q90
        ),

        # Pairwise cosine distance
        "pairwise_distance_mean": float(
            np.mean(pairwise_distances)
        ),
        "pairwise_distance_sd": _sample_sd(
            pairwise_distances
        ),
        "pairwise_distance_q10": float(
            dist_q10
        ),
        "pairwise_distance_median": float(
            dist_median
        ),
        "pairwise_distance_q90": float(
            dist_q90
        ),
        "pairwise_semantic_span": float(
            dist_q90 - dist_q10
        ),

        # Local-neighborhood structure
        "nearest_neighbor_distance_mean": float(
            np.mean(nearest_neighbor_distances)
        ),
        "nearest_neighbor_distance_median": float(
            np.median(nearest_neighbor_distances)
        ),
        "knn_distance_median": float(
            np.median(kth_neighbor_distances)
        ),

        # Covariance-spectrum structure
        "maximum_available_rank": int(
            maximum_available_rank
        ),
        "numerical_rank": int(
            numerical_rank
        ),
        "total_variance": (
            total_variance
        ),
        "spectral_entropy": (
            spectral_entropy
        ),
        "spectral_entropy_normalized": (
            spectral_entropy_normalized
        ),
        "spectral_effective_rank": (
            spectral_effective_rank
        ),
        "participation_ratio": (
            participation_ratio
        ),
        "pc1_variance_share": (
            pc1_variance_share
        ),
    }

    # Retain arrays separately so they do not clutter the DataFrame.
    spectrum = {
        "covariance_eigenvalues": eigenvalues,
        "normalized_eigenvalues": normalized_eigenvalues,
    }

    # ------------------------------------------------------------
    # 9. Mathematical consistency checks
    # ------------------------------------------------------------

    np.testing.assert_allclose(
        scalar_metrics["pairwise_distance_mean"],
        total_variance,
        atol=1e-10,
    )

    np.testing.assert_allclose(
        scalar_metrics[
            "pairwise_cosine_similarity_mean"
        ],
        1.0
        - scalar_metrics["pairwise_distance_mean"],
        atol=1e-10,
    )

    np.testing.assert_allclose(
        scalar_metrics["pairwise_distance_mean"],
        n / (n - 1)
        * (1.0 - centroid_resultant_length**2),
        atol=1e-10,
    )

    if np.isfinite(radial_distance_mean):
        np.testing.assert_allclose(
            radial_distance_mean,
            centroid_dispersion_cosine,
            atol=1e-10,
        )

    return scalar_metrics, spectrum


# ================================================================
# Calculate metrics for every conversation
# ================================================================

geometry_records = []
conversation_spectra = {}

group_columns = [
    "dataset",
    "conversation_id",
]

for group_key, group in geometry_utterances.groupby(
    group_columns,
    sort=False,
    observed=True,
):
    dataset, conversation_id = group_key

    # embedding_idx maps each utterance row to E_utterances.
    embedding_indices = group[
        "embedding_idx"
    ].to_numpy(dtype=int)

    conversation_embeddings = E_utterances[
        embedding_indices
    ]

    scalar_metrics, spectrum = (
        calculate_conversation_geometry(
            conversation_embeddings,
            n_words=group["n_words"].sum(),
            n_turns=group["turn_id"].nunique(),
            k=5,
        )
    )

    row = {
        "dataset": dataset,
        "conversation_id": conversation_id,
    }

    # Preserve useful conversation metadata where available.
    for column in [
        "video_id",
        "file_number",
        "level",
        "level_label",
    ]:
        if column in group.columns:
            row[column] = group[column].iloc[0]

    row.update(scalar_metrics)
    geometry_records.append(row)

    conversation_spectra[
        (dataset, conversation_id)
    ] = spectrum


conversation_geometry = pd.DataFrame(
    geometry_records
)

sort_columns = [
    column
    for column in [
        "level",
        "dataset",
        "conversation_id",
    ]
    if column in conversation_geometry.columns
]

conversation_geometry = (
    conversation_geometry
    .sort_values(sort_columns)
    .reset_index(drop=True)
)

print(
    "Conversations analyzed:",
    len(conversation_geometry),
)

print(
    "Geometry table shape:",
    conversation_geometry.shape,
)

display(conversation_geometry)

Conversations analyzed: 105
Geometry table shape: (105, 39)


,dataset,conversation_id,video_id,file_number,level,level_label,n_utterances,n_pairs,n_words,n_turns,...,nearest_neighbor_distance_median,knn_distance_median,maximum_available_rank,numerical_rank,total_variance,spectral_entropy,spectral_entropy_normalized,spectral_effective_rank,participation_ratio,pc1_variance_share
0,wired,wired_Talia_12,talia,12,0,child,27,351,315,14,...,0.501682,0.730119,26,26,0.825017,2.993523,0.918795,19.955860,16.181812,0.133915
1,wired,wired_astro_12,astro,12,0,child,56,1540,647,23,...,0.399942,0.595853,55,55,0.816119,3.508485,0.875516,33.397620,22.543430,0.121501
2,wired,wired_blockchain_12,blockchain,12,0,child,20,190,191,9,...,0.433820,0.788989,19,19,0.832669,2.693383,0.914736,14.781598,12.488393,0.148911
3,wired,wired_crispr_12,crispr,12,0,child,21,210,152,12,...,0.403874,0.757450,20,19,0.812852,2.639855,0.881205,14.011166,11.503853,0.174152
4,wired,wired_dimension_12,dimension,12,0,child,89,3916,689,52,...,0.310556,0.546307,88,80,0.795924,3.683957,0.822801,39.803569,22.846723,0.144173
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,wired,wired_nuclear_16,nuclear,16,4,expert,44,946,849,11,...,0.445602,0.629145,43,43,0.791545,3.391523,0.901713,29.711170,21.622795,0.128743
101,wired,wired_sensing_16,sensing,16,4,expert,56,1540,882,27,...,0.486627,0.652179,55,50,0.829185,3.464621,0.864570,31.964335,21.181876,0.141975
102,wired,wired_sleep_16,sleep,16,4,expert,31,465,823,13,...,0.341957,0.446537,30,30,0.693088,2.986731,0.878141,19.820781,13.797187,0.177302
103,wired,wired_time_16,time,16,4,expert,63,1953,1182,23,...,0.483401,0.622610,62,62,0.801034,3.757115,0.910345,42.824685,31.911495,0.083664


In [16]:
conversation_spectra[
    (dataset, conversation_id)
]["normalized_eigenvalues"]

array([0.13867053, 0.06422416, 0.06012775, 0.04561799, 0.03894875,
       0.03477304, 0.03209123, 0.03086069, 0.02856981, 0.02604829,
       0.0237671 , 0.022569  , 0.02142719, 0.01848028, 0.01768911,
       0.01675951, 0.01649905, 0.01585879, 0.01512339, 0.01490982,
       0.01363454, 0.01311112, 0.01279547, 0.01237183, 0.01148387,
       0.01108127, 0.01037747, 0.01029258, 0.01004799, 0.00962358,
       0.00943986, 0.0086673 , 0.00837013, 0.00817452, 0.00786843,
       0.0074397 , 0.00736857, 0.0069895 , 0.00669382, 0.00651688,
       0.00629284, 0.00603892, 0.00578112, 0.00563007, 0.00538437,
       0.00514307, 0.00489986, 0.00468985, 0.00442706, 0.00428041,
       0.00424491, 0.0041    , 0.00374   , 0.00369798, 0.00363428,
       0.00357844, 0.00342371, 0.00314444, 0.0029595 , 0.00284897,
       0.0026051 , 0.00255609, 0.00243477, 0.00228985, 0.00218601,
       0.00215451, 0.00196746, 0.00190469, 0.00182464, 0.00173968,
       0.00161436, 0.00159506, 0.00138132, 0.00135497, 0.00129

In [17]:
import matplotlib.pyplot as plt
import seaborn as sns

level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

def plot_metric_distribution(df, metric, ylabel=None):
    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="level_label",
        y=metric,
        order=level_order,
        showfliers=False,
        color="lightgray",
    )

    sns.stripplot(
        data=df,
        x="level_label",
        y=metric,
        order=level_order,
        color="black",
        alpha=0.6,
        size=4,
        jitter=0.18,
    )

    plt.xlabel("Partner expertise → smaller expertise gap")
    plt.ylabel(ylabel or metric)
    plt.title(ylabel or metric)
    plt.tight_layout()
    plt.show()

In [18]:
conversation_geometry.columns


Index(['dataset', 'conversation_id', 'video_id', 'file_number', 'level',
       'level_label', 'n_utterances', 'n_pairs', 'n_words', 'n_turns',
       'embedding_dimension', 'k_used', 'pairwise_cosine_similarity_mean',
       'pairwise_cosine_similarity_sd', 'pairwise_cosine_similarity_q10',
       'pairwise_cosine_similarity_median', 'pairwise_cosine_similarity_q90',
       'centroid_resultant_length', 'centroid_dispersion_cosine',
       'radial_distance_mean_check', 'radial_distance_median',
       'radial_distance_q90', 'pairwise_distance_mean', 'pairwise_distance_sd',
       'pairwise_distance_q10', 'pairwise_distance_median',
       'pairwise_distance_q90', 'pairwise_semantic_span',
       'nearest_neighbor_distance_mean', 'nearest_neighbor_distance_median',
       'knn_distance_median', 'maximum_available_rank', 'numerical_rank',
       'total_variance', 'spectral_entropy', 'spectral_entropy_normalized',
       'spectral_effective_rank', 'participation_ratio', 'pc1_variance_shar

In [19]:
plot_metric_distribution(
    conversation_geometry,
    "participation_ratio"
)

In [25]:
conversation_geometry

,dataset,conversation_id,video_id,file_number,level,level_label,n_utterances,n_pairs,n_words,n_turns,...,nearest_neighbor_distance_median,knn_distance_median,maximum_available_rank,numerical_rank,total_variance,spectral_entropy,spectral_entropy_normalized,spectral_effective_rank,participation_ratio,pc1_variance_share
0,wired,wired_Talia_12,talia,12,0,child,27,351,315,14,...,0.501682,0.730119,26,26,0.825017,2.993523,0.918795,19.955860,16.181813,0.133915
1,wired,wired_astro_12,astro,12,0,child,56,1540,647,23,...,0.399942,0.595853,55,55,0.816119,3.508485,0.875516,33.397620,22.543430,0.121501
2,wired,wired_blockchain_12,blockchain,12,0,child,20,190,191,9,...,0.433820,0.788989,19,19,0.832669,2.693383,0.914736,14.781598,12.488393,0.148911
3,wired,wired_crispr_12,crispr,12,0,child,21,210,152,12,...,0.403874,0.757450,20,18,0.812852,2.639855,0.881205,14.011166,11.503852,0.174152
4,wired,wired_dimension_12,dimension,12,0,child,89,3916,689,52,...,0.310556,0.546307,88,81,0.795924,3.683957,0.822801,39.803569,22.846722,0.144173
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,wired,wired_nuclear_16,nuclear,16,4,expert,44,946,849,11,...,0.445602,0.629145,43,43,0.791545,3.391523,0.901713,29.711170,21.622795,0.128743
101,wired,wired_sensing_16,sensing,16,4,expert,56,1540,882,27,...,0.486627,0.652179,55,50,0.829185,3.464621,0.864570,31.964335,21.181875,0.141975
102,wired,wired_sleep_16,sleep,16,4,expert,31,465,823,13,...,0.341957,0.446537,30,30,0.693088,2.986731,0.878141,19.820781,13.797187,0.177302
103,wired,wired_time_16,time,16,4,expert,63,1953,1182,23,...,0.483401,0.622610,62,62,0.801034,3.757115,0.910345,42.824685,31.911495,0.083664


In [26]:
def plot_metric_trajectories(df, metric, ylabel=None):
    fig, ax = plt.subplots(figsize=(8, 5))

    # One thin trajectory per topic
    sns.lineplot(
        data=df,
        x="level",
        y=metric,
        units="video_id",
        estimator=None,
        color="gray",
        alpha=0.3,
        linewidth=1,
        marker="o",
        markersize=3,
        ax=ax,
    )

    # Overall mean and 95% bootstrap CI
    sns.lineplot(
        data=df,
        x="level",
        y=metric,
        estimator="mean",
        errorbar=("ci", 95),
        color="black",
        linewidth=3,
        marker="o",
        markersize=7,
        ax=ax,
    )

    ax.set_xticks(range(5))
    ax.set_xticklabels(level_order)
    ax.set_xlabel("Partner expertise → smaller expertise gap")
    ax.set_ylabel(ylabel or metric)
    ax.set_title(ylabel or metric)

    plt.tight_layout()
    plt.show()

In [27]:
plot_metric_trajectories(
    conversation_geometry,
    "nearest_neighbor_distance_mean",
    "Mean nearest-neighbor distance",
)

In [29]:
def plot_sample_size_diagnostic(df, metric, ylabel=None):
    plt.figure(figsize=(7, 5))

    sns.scatterplot(
        data=df,
        x="n_utterances",
        y=metric,
        hue="level_label",
        hue_order=level_order,
        alpha=0.8,
    )

    sns.regplot(
        data=df,
        x="n_utterances",
        y=metric,
        scatter=False,
        color="black",
        line_kws={"linestyle": "--"},
    )

    plt.xlabel("Number of utterances")
    plt.ylabel(ylabel or metric)
    plt.title(f"{ylabel or metric} versus conversation size")
    plt.tight_layout()
    plt.show()

In [30]:
plot_sample_size_diagnostic(
    conversation_geometry,
    "participation_ratio",
    "Participation ratio",
)

plot_sample_size_diagnostic(
    conversation_geometry,
    "spectral_effective_rank",
    "Spectral effective rank",
)

In [31]:
plot_metric_distribution(
    conversation_geometry,
    "nearest_neighbor_distance_mean",
    "Mean nearest-neighbor distance",
)

plot_metric_distribution(
    conversation_geometry,
    "spectral_entropy_normalized",
    "Normalized spectral entropy",
)

plot_metric_distribution(
    conversation_geometry,
    "pc1_variance_share",
    "PC1 variance share",
)

In [33]:
conversation_geometry.columns

Index(['dataset', 'conversation_id', 'video_id', 'file_number', 'level',
       'level_label', 'n_utterances', 'n_pairs', 'n_words', 'n_turns',
       'embedding_dimension', 'k_used', 'pairwise_cosine_similarity_mean',
       'pairwise_cosine_similarity_sd', 'pairwise_cosine_similarity_q10',
       'pairwise_cosine_similarity_median', 'pairwise_cosine_similarity_q90',
       'centroid_resultant_length', 'centroid_dispersion_cosine',
       'radial_distance_mean_check', 'radial_distance_median',
       'radial_distance_q90', 'pairwise_distance_mean', 'pairwise_distance_sd',
       'pairwise_distance_q10', 'pairwise_distance_median',
       'pairwise_distance_q90', 'pairwise_semantic_span',
       'nearest_neighbor_distance_mean', 'nearest_neighbor_distance_median',
       'knn_distance_median', 'maximum_available_rank', 'numerical_rank',
       'total_variance', 'spectral_entropy', 'spectral_entropy_normalized',
       'spectral_effective_rank', 'participation_ratio', 'pc1_variance_shar

In [34]:
plot_metric_distribution(
    conversation_geometry,
    "participation_ratio"
)

In [20]:
plot_metric_distribution(
    conversation_geometry,
    "radial_distance_q90"
)

# multilevel modeling for each metric

model each metric using a mixed effect(multilevel) model. (https://pmc.ncbi.nlm.nih.gov/articles/PMC10171296/)


gap scores: 12->4 13->3 14->2 15->1 16->0

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

analysis_df = conversation_geometry.copy()

gap_map = {
    "expert": 0,
    "graduate": 1,
    "undergraduate": 2,
    "teenager": 3,
    "child": 4,
}

analysis_df["gap_score"] = (
    analysis_df["level_label"]
    .map(gap_map)
    .astype(float)
)

# Centering makes the intercept represent the middle expertise condition.
analysis_df["gap_c"] = (
    analysis_df["gap_score"]
    - analysis_df["gap_score"].mean()
)

# Use a composite identifier in case video_id is repeated across datasets.
analysis_df["video_group"] = (
    analysis_df["dataset"].astype(str)
    + "::"
    + analysis_df["video_id"].astype(str)
)

# Length variable for sensitivity analyses
analysis_df["log_n_utterances"] = np.log(
    analysis_df["n_utterances"]
)

analysis_df["log_n_utterances_c"] = (
    analysis_df["log_n_utterances"]
    - analysis_df["log_n_utterances"].mean()
)

analysis_df["level_label"] = pd.Categorical(
    analysis_df["level_label"],
    categories=[
        "expert",
        "graduate",
        "undergraduate",
        "teenager",
        "child",
    ],
    ordered=True,
)

In [ ]:
#standardize outcomes
def prepare_metric(df, metric):
    model_df = df[
        [
            metric,
            "gap_score",
            "gap_c",
            "level_label",
            "video_group",
            "n_utterances",
            "log_n_utterances_c",
        ]
    ].dropna().copy()

    model_df["outcome_z"] = (
        model_df[metric] - model_df[metric].mean()
    ) / model_df[metric].std(ddof=1)

    return model_df

-> betagap represents SD change in the metric per gap level

## model 0 (null model)

separates vaience into 1: between video variance 2: within video residual variance

In [ ]:
metric = "pairwise_distance_mean"
model_df = prepare_metric(analysis_df, metric)

model_0 = smf.mixedlm(
    "outcome_z ~ 1",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_0.summary())

In [ ]:
#intraclass correlation
between_video_variance = model_0.cov_re.iloc[0, 0]
residual_variance = model_0.scale

icc = (
    between_video_variance
    / (between_video_variance + residual_variance)
)

print("ICC:", icc)

icc: % unexplained variation that occurs between videos. 5 is good


## model 1 : linear expertisegap model

In [ ]:
model_1 = smf.mixedlm(
    "outcome_z ~ gap_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_1.summary())

In [ ]:
beta_gap = model_1.fe_params["gap_c"]
ci_gap = model_1.conf_int().loc["gap_c"]

print("Gap coefficient:", beta_gap)
print("95% CI:", tuple(ci_gap))
print("Predicted child–expert difference:", 4 * beta_gap)

## model 2: length adjusted model

In [ ]:
model_2 = smf.mixedlm(
    "outcome_z ~ gap_c + log_n_utterances_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_2.summary())

In [ ]:
#compare gap coefficients between length adjusted and not
comparison = pd.DataFrame({
    "model": [
        "Unadjusted",
        "Length-adjusted",
    ],
    "gap_beta": [
        model_1.fe_params["gap_c"],
        model_2.fe_params["gap_c"],
    ],
    "gap_p": [
        model_1.pvalues["gap_c"],
        model_2.pvalues["gap_c"],
    ],
})

display(comparison)

length probably doesnt have too big of an effect

## model 3: categorical expertise model

asks if each partner level differs from expert condition WITHOUT imposing linear progression

In [ ]:
model_3 = smf.mixedlm(
    (
        "outcome_z ~ "
        "C(level_label, Treatment(reference='expert'))"
    ),
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_3.summary())

In [ ]:
from scipy.stats import chi2

model_linear_ml = smf.mixedlm(
    "outcome_z ~ gap_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=False,
    method="lbfgs",
)

model_categorical_ml = smf.mixedlm(
    (
        "outcome_z ~ "
        "C(level_label, Treatment(reference='expert'))"
    ),
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=False,
    method="lbfgs",
)

lr_statistic = 2 * (
    model_categorical_ml.llf
    - model_linear_ml.llf
)

df_difference = (
    len(model_categorical_ml.fe_params)
    - len(model_linear_ml.fe_params)
)

lr_pvalue = chi2.sf(
    lr_statistic,
    df_difference,
)

print("Linear AIC:", model_linear_ml.aic)
print("Categorical AIC:", model_categorical_ml.aic)
print("Likelihood-ratio statistic:", lr_statistic)
print("Degrees of freedom:", df_difference)
print("p-value:", lr_pvalue)

AIC values are similar - USE LINEAR MODEL FOR PARSIMONY

# model all selected metrics

In [ ]:
# set of metrics to be modeled
metrics_to_model = [
    "pairwise_distance_mean",
    "pairwise_distance_sd",
    "radial_distance_q90",
    "nearest_neighbor_distance_mean",
    "knn_distance_median",
    "spectral_entropy_normalized",
    "participation_ratio",
    "pc1_variance_share",
]

linear random intercept model for each metric

In [ ]:
def fit_metric_model(
    df,
    metric,
    adjust_for_length=False,
    random_slope=False,
):
    model_df = prepare_metric(df, metric)

    formula = "outcome_z ~ gap_c"

    if adjust_for_length:
        formula += " + log_n_utterances_c"

    re_formula = (
        "~gap_c"
        if random_slope
        else "1"
    )

    result = smf.mixedlm(
        formula,
        data=model_df,
        groups=model_df["video_group"],
        re_formula=re_formula,
    ).fit(
        reml=True,
        method="lbfgs",
    )

    return result

In [ ]:
#run models. get gap effects
model_results = {}
summary_records = []

for metric in metrics_to_model:
    result = fit_metric_model(
        analysis_df,
        metric,
        adjust_for_length=False,
        random_slope=False,
    )

    model_results[metric] = result

    ci = result.conf_int().loc["gap_c"]

    summary_records.append({
        "metric": metric,
        "gap_beta_standardized":
            result.fe_params["gap_c"],
        "gap_se":
            result.bse["gap_c"],
        "ci_lower":
            ci.iloc[0],
        "ci_upper":
            ci.iloc[1],
        "p_value":
            result.pvalues["gap_c"],
        "converged":
            result.converged,
        "child_expert_difference_sd":
            4 * result.fe_params["gap_c"],
    })

model_summary = pd.DataFrame(summary_records)

display(model_summary)

In [ ]:
#correct for multiple testing
from statsmodels.stats.multitest import multipletests

model_summary["p_fdr_bh"] = multipletests(
    model_summary["p_value"],
    method="fdr_bh",
)[1]

model_summary["significant_fdr_05"] = (
    model_summary["p_fdr_bh"] < 0.05
)

model_summary = model_summary.sort_values(
    "p_fdr_bh"
).reset_index(drop=True)

display(model_summary)

# diagnostics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

result = model_results["pairwise_distance_mean"]

residuals = result.resid
fitted = result.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.scatterplot(
    x=fitted,
    y=residuals,
    ax=axes[0],
)

axes[0].axhline(
    0,
    color="black",
    linestyle="--",
)

axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals versus fitted")

stats.probplot(
    residuals,
    dist="norm",
    plot=axes[1],
)

axes[1].set_title("Residual Q–Q plot")

plt.tight_layout()
plt.show()

In [ ]:
model_summary

larger gap -> one dominant semantic direction(this makes sense), less even spectral variation,

# Speaker–speaker centroid geometry (corrected)

This section measures the cosine distance between the explaining expert's mean utterance embedding and the conversation partner's mean utterance embedding in each conversation.

The earlier version returned only `NaN` because the notebook stores `speaker` as `"A"`/`"B"`, while the function searched for `"Speaker A"`/`"Speaker B"`. This corrected version uses the canonical `speaker_role` values already created by the notebook: `"expert"` and `"partner"`.

Run the notebook through the construction of `utterances`, `E_utterances`, and `conversation_geometry` before running this section.

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import cosine

required_objects = [
    "utterances",
    "E_utterances",
    "conversation_geometry",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the earlier notebook cells first. Missing: "
        + ", ".join(missing_objects)
    )

embedding_matrix = np.asarray(E_utterances)

print("utterances shape:", utterances.shape)
print("embedding matrix shape:", embedding_matrix.shape)
print("speaker values:")
print(utterances["speaker"].value_counts(dropna=False))
print("speaker_role values:")
print(utterances["speaker_role"].value_counts(dropna=False))

utterances shape: (6381, 23)
embedding matrix shape: (6381, 384)
speaker values:
speaker
A    4063
B    2318
Name: count, dtype: int64
speaker_role values:
speaker_role
expert     4063
partner    2318
Name: count, dtype: int64


In [29]:
# Validate the explicit mapping between table rows and embedding rows.
embedding_indices = (
    utterances["embedding_idx"]
    .dropna()
    .astype(int)
    .to_numpy()
)

if len(embedding_indices) == 0:
    raise ValueError("utterances contains no embedding_idx values.")

if embedding_indices.min() < 0:
    raise ValueError("embedding_idx contains a negative value.")

if embedding_indices.max() >= embedding_matrix.shape[0]:
    raise ValueError(
        f"Maximum embedding_idx is {embedding_indices.max()}, "
        f"but E_utterances has {embedding_matrix.shape[0]} rows."
    )

if utterances["embedding_idx"].duplicated().any():
    raise ValueError("embedding_idx must be unique in utterances.")

if not np.isfinite(embedding_matrix).all():
    raise ValueError("E_utterances contains non-finite values.")

print("Embedding alignment checks passed.")

Embedding alignment checks passed.


In [30]:
# Use the same primary subset as the earlier conversation geometry.
# This handles both Boolean and string representations safely.
include_primary = (
    utterances["include_geometry_primary"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

geometry_utterances = utterances.loc[include_primary].copy()

# Normalize the already-canonical roles defensively.
geometry_utterances["_centroid_role"] = (
    geometry_utterances["speaker_role"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "learner": "partner",
        "speaker a": "expert",
        "a": "expert",
        "speaker b": "partner",
        "b": "partner",
    })
)

unexpected_roles = sorted(
    set(geometry_utterances["_centroid_role"].dropna())
    - {"expert", "partner"}
)

if unexpected_roles:
    raise ValueError(
        "Unrecognized speaker roles: "
        + repr(unexpected_roles)
    )

role_counts = (
    geometry_utterances
    .groupby(
        ["dataset", "conversation_id", "_centroid_role"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
)

missing_role_conversations = role_counts[
    (role_counts.get("expert", 0) == 0)
    | (role_counts.get("partner", 0) == 0)
]

display(role_counts.head())

if not missing_role_conversations.empty:
    display(missing_role_conversations)
    raise ValueError(
        "At least one primary-geometry conversation lacks an expert "
        "or partner. See the displayed table."
    )

print(
    f"Validated {len(role_counts)} conversations; "
    f"all contain both roles."
)

_centroid_role           expert  partner
dataset conversation_id                 
wired   wired_Talia_12       19        1
        wired_Talia_13       37        8
        wired_Talia_14       29        9
        wired_Talia_15       27        8
        wired_Talia_16       16       27

Validated 105 conversations; all contain both roles.


In [33]:
def calculate_speaker_centroid_distance(
    conversation_utterances,
    embeddings,
    role_column="_centroid_role",
    embedding_index_column="embedding_idx",
):
    """Return expert–partner centroid distances for one conversation."""
    expert_indices = (
        conversation_utterances.loc[
            conversation_utterances[role_column].eq("expert"),
            embedding_index_column,
        ]
        .dropna()
        .astype(int)
        .to_numpy()
    )

    partner_indices = (
        conversation_utterances.loc[
            conversation_utterances[role_column].eq("partner"),
            embedding_index_column,
        ]
        .dropna()
        .astype(int)
        .to_numpy()
    )

    if len(expert_indices) == 0 or len(partner_indices) == 0:
        raise ValueError(
            "Each conversation must contain both expert and partner utterances."
        )

    expert_centroid = embeddings[expert_indices].mean(axis=0)
    partner_centroid = embeddings[partner_indices].mean(axis=0)

    return {
        "expert_n_utterances": int(len(expert_indices)),
        "partner_n_utterances": int(len(partner_indices)),
        "speaker_centroid_cosine_distance": float(
            cosine(expert_centroid, partner_centroid)
        ),
        "speaker_centroid_euclidean_distance": float(
            np.linalg.norm(expert_centroid - partner_centroid)
        ),
    }


# Test one conversation before applying the function to all conversations.
first_key, first_group = next(iter(
    geometry_utterances.groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    )
))

print("Test conversation:", first_key)
print(
    calculate_speaker_centroid_distance(
        first_group,
        embedding_matrix,
    )
)

Test conversation: ('wired', 'wired_Talia_12')
{'expert_n_utterances': 19, 'partner_n_utterances': 1, 'speaker_centroid_cosine_distance': 0.46671104431152344, 'speaker_centroid_euclidean_distance': 0.8467584252357483}


In [34]:
speaker_centroid_records = []

for (dataset, conversation_id), group in (
    geometry_utterances.groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    )
):
    result = calculate_speaker_centroid_distance(
        conversation_utterances=group,
        embeddings=embedding_matrix,
    )

    speaker_centroid_records.append({
        "dataset": dataset,
        "conversation_id": conversation_id,
        **result,
    })

speaker_centroid_geometry = pd.DataFrame(speaker_centroid_records)

if speaker_centroid_geometry.empty:
    raise ValueError("No conversation-level centroid results were created.")

distance_columns = [
    "speaker_centroid_cosine_distance",
    "speaker_centroid_euclidean_distance",
]

if speaker_centroid_geometry[distance_columns].isna().any().any():
    raise ValueError("Centroid calculation unexpectedly produced NaN.")

display(speaker_centroid_geometry.head())
display(speaker_centroid_geometry[distance_columns].describe())

,dataset,conversation_id,expert_n_utterances,partner_n_utterances,speaker_centroid_cosine_distance,speaker_centroid_euclidean_distance
0,wired,wired_Talia_12,19,1,0.466711,0.846758
1,wired,wired_Talia_13,37,8,0.463972,0.469025
2,wired,wired_Talia_14,29,9,0.239723,0.367751
3,wired,wired_Talia_15,27,8,0.233536,0.395115
4,wired,wired_Talia_16,16,27,0.101308,0.287101


,speaker_centroid_cosine_distance,speaker_centroid_euclidean_distance
count,105.000000,105.000000
mean,0.241214,0.369261
std,0.114057,0.122493
min,0.075563,0.185967
25%,0.160204,0.297549
50%,0.230380,0.348282
75%,0.281255,0.406411
max,0.798183,1.027541


In [35]:
# Merge by dataset and conversation_id to avoid accidental cross-dataset matches.
centroid_output_columns = [
    "expert_n_utterances",
    "partner_n_utterances",
    "speaker_centroid_cosine_distance",
    "speaker_centroid_euclidean_distance",
]

conversation_geometry = conversation_geometry.drop(
    columns=[
        column for column in centroid_output_columns
        if column in conversation_geometry.columns
    ],
    errors="ignore",
)

conversation_geometry = conversation_geometry.merge(
    speaker_centroid_geometry,
    on=["dataset", "conversation_id"],
    how="left",
    validate="one_to_one",
)

missing_after_merge = conversation_geometry[
    "speaker_centroid_cosine_distance"
].isna()

print(
    f"Merged centroid metrics into {len(conversation_geometry)} conversations."
)
print("Missing after merge:", int(missing_after_merge.sum()))

if missing_after_merge.any():
    display(
        conversation_geometry.loc[
            missing_after_merge,
            ["dataset", "conversation_id", "level_label"],
        ]
    )
    raise ValueError(
        "Some conversation_geometry rows did not match centroid results."
    )

display(
    conversation_geometry[
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level_label",
            "expert_n_utterances",
            "partner_n_utterances",
            "speaker_centroid_cosine_distance",
        ]
    ].head()
)

Merged centroid metrics into 105 conversations.
Missing after merge: 0


,dataset,video_id,conversation_id,level_label,expert_n_utterances,partner_n_utterances,speaker_centroid_cosine_distance
0,wired,talia,wired_Talia_12,child,19,1,0.466711
1,wired,astro,wired_astro_12,child,35,11,0.170952
2,wired,blockchain,wired_blockchain_12,child,11,4,0.368993
3,wired,crispr,wired_crispr_12,child,12,1,0.798183
4,wired,dimension,wired_dimension_12,child,46,15,0.241155


## Visualizations

Higher cosine distance means the two speakers' average semantic positions are farther apart. The matched-lines plot is especially important because it shows whether the pattern repeats within topics rather than being driven only by differences among topics.

In [36]:
level_lookup = (
    conversation_geometry[["level", "level_label"]]
    .dropna()
    .drop_duplicates()
    .sort_values("level")
)

level_order = level_lookup["level_label"].tolist()
level_values = level_lookup["level"].tolist()

plot_data = conversation_geometry.dropna(
    subset=["level_label", "speaker_centroid_cosine_distance"]
).copy()

if plot_data.empty:
    raise ValueError("No finite centroid distances are available to plot.")

palette = expertise_palette(level_order)

display(
    plot_data.groupby("level_label", observed=True)[
        "speaker_centroid_cosine_distance"
    ].agg(["count", "mean", "std", "median"])
)

,count,mean,std,median
level_label,,,,
child,21,0.314863,0.145261,0.272044
expert,21,0.161380,0.050207,0.151573
graduate,21,0.197487,0.061038,0.205028
teenager,21,0.306119,0.122691,0.267978
undergraduate,21,0.226222,0.078689,0.212725


In [37]:
# Distribution by expertise level, using Matplotlib for the box layer.
rng = np.random.default_rng(2026)
groups = [
    plot_data.loc[
        plot_data["level_label"].eq(level),
        "speaker_centroid_cosine_distance",
    ].to_numpy()
    for level in level_order
]

fig, ax = plt.subplots(figsize=(9, 6))

box = ax.boxplot(
    groups,
    labels=level_order,
    showfliers=False,
    patch_artist=True,
)

for patch, level in zip(box["boxes"], level_order):
    patch.set_facecolor(palette[level])
    patch.set_alpha(0.55)

for position, (level, values) in enumerate(
    zip(level_order, groups),
    start=1,
):
    x = position + rng.uniform(-0.12, 0.12, size=len(values))
    ax.scatter(x, values, color="black", alpha=0.45, s=22, zorder=3)

ax.set_xlabel("Conversation partner expertise")
ax.set_ylabel("Expert–partner centroid cosine distance")
ax.set_title("Semantic distance between speaker centroids")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

/var/folders/1z/hv4_qzyn6556dk5hqb4k6tdh0000gn/T/ipykernel_44435/2189465919.py:13: MatplotlibDeprecationWarning:

The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.



In [ ]:
zsx

In [ ]:
# Matched within-topic trajectories.
fig, ax = plt.subplots(figsize=(9, 6))

for (_, video_id), group in plot_data.groupby(
    ["dataset", "video_id"],
    observed=True,
):
    group = group.sort_values("level")
    ax.plot(
        group["level"],
        group["speaker_centroid_cosine_distance"],
        color="gray",
        alpha=0.3,
        linewidth=1,
    )

level_summary = (
    plot_data.groupby(["level", "level_label"], observed=True)
    ["speaker_centroid_cosine_distance"]
    .agg(["mean", "sem"])
    .reset_index()
    .sort_values("level")
)

ax.errorbar(
    level_summary["level"],
    level_summary["mean"],
    yerr=1.96 * level_summary["sem"],
    color="black",
    marker="o",
    linewidth=2.5,
    capsize=4,
    label="Mean ± 95% normal CI",
)

ax.set_xticks(level_values)
ax.set_xticklabels(level_order, rotation=20)
ax.set_xlabel("Conversation partner expertise")
ax.set_ylabel("Expert–partner centroid cosine distance")
ax.set_title("Within-topic change in speaker-centroid separation")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Topic-by-level heatmap.
centroid_heatmap = (
    plot_data.pivot_table(
        index=["dataset", "video_id"],
        columns="level_label",
        values="speaker_centroid_cosine_distance",
        aggfunc="first",
    )
    .reindex(columns=level_order)
)

fig, ax = plt.subplots(
    figsize=(9, max(5, 0.35 * len(centroid_heatmap)))
)

sns.heatmap(
    centroid_heatmap,
    cmap="viridis",
    linewidths=0.3,
    cbar_kws={"label": "Centroid cosine distance"},
    ax=ax,
)

ax.set_xlabel("Conversation partner expertise")
ax.set_ylabel("Topic")
ax.set_title("Speaker-centroid separation by topic and expertise")
plt.tight_layout()
plt.show()

In [ ]:
# Save reusable conversation-level results.
from pathlib import Path

OUTPUT_DIR = Path("analysis_exports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

speaker_centroid_geometry.to_csv(
    OUTPUT_DIR / "speaker_centroid_geometry.csv",
    index=False,
)

conversation_geometry.to_csv(
    OUTPUT_DIR / "conversation_geometry_with_speaker_centroids.csv",
    index=False,
)

print("Saved centroid results to:", OUTPUT_DIR.resolve())

# Leave-one-out radial geometry

In [ ]:
def leave_one_out_radial_metrics(X):
    X = np.asarray(X, dtype=float)

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_unit = X / np.maximum(norms, 1e-12)

    n = len(X_unit)

    if n < 3:
        return {
            "radial_loo_q10": np.nan,
            "radial_loo_median": np.nan,
            "radial_loo_q90": np.nan,
        }

    total_vector = X_unit.sum(axis=0)
    radial_distances = []

    for i in range(n):
        loo_centroid = (
            total_vector - X_unit[i]
        ) / (n - 1)

        centroid_norm = np.linalg.norm(
            loo_centroid
        )

        if centroid_norm <= 1e-12:
            radial_distances.append(np.nan)
            continue

        centroid_direction = (
            loo_centroid / centroid_norm
        )

        distance = (
            1.0
            - X_unit[i] @ centroid_direction
        )

        radial_distances.append(distance)

    radial_distances = np.asarray(
        radial_distances
    )

    return {
        "radial_loo_q10": np.nanquantile(
            radial_distances, 0.10
        ),
        "radial_loo_median": np.nanmedian(
            radial_distances
        ),
        "radial_loo_q90": np.nanquantile(
            radial_distances, 0.90
        ),
    }

In [ ]:
radial_loo_records = []

for (dataset, conversation_id), group in geometry_utterances.groupby(
    ["dataset", "conversation_id"],
    observed=True,
    sort=False,
):
    embedding_indices = group[
        "embedding_idx"
    ].to_numpy(dtype=int)

    metrics = leave_one_out_radial_metrics(
        E_utterances[embedding_indices]
    )

    radial_loo_records.append({
        "dataset": dataset,
        "video_id": group["video_id"].iloc[0],
        "conversation_id": conversation_id,
        "level": group["level"].iloc[0],
        "level_label": group["level_label"].iloc[0],
        "n_utterances": len(group),
        **metrics,
    })

radial_loo_summary = pd.DataFrame(
    radial_loo_records
)

display(radial_loo_summary.head())

In [ ]:
import numpy as np
import pandas as pd

def calculate_loo_radial_metrics(X):
    X = np.asarray(X, dtype=float)

    X_unit = X / np.maximum(
        np.linalg.norm(X, axis=1, keepdims=True),
        1e-12,
    )

    n = len(X_unit)

    if n < 3:
        return {
            "radial_loo_q10": np.nan,
            "radial_loo_median": np.nan,
            "radial_loo_q90": np.nan,
        }

    total_vector = X_unit.sum(axis=0)
    radial_distances = np.full(n, np.nan)

    for i in range(n):
        loo_centroid = (
            total_vector - X_unit[i]
        ) / (n - 1)

        centroid_norm = np.linalg.norm(
            loo_centroid
        )

        if centroid_norm > 1e-12:
            centroid_direction = (
                loo_centroid / centroid_norm
            )

            radial_distances[i] = np.clip(
                1.0
                - X_unit[i] @ centroid_direction,
                0.0,
                2.0,
            )

    return {
        "radial_loo_q10": np.nanquantile(
            radial_distances, 0.10
        ),
        "radial_loo_median": np.nanmedian(
            radial_distances
        ),
        "radial_loo_q90": np.nanquantile(
            radial_distances, 0.90
        ),
    }


radial_records = []

for (dataset, conversation_id), group in (
    geometry_utterances.groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    )
):
    embedding_indices = group[
        "embedding_idx"
    ].to_numpy(dtype=int)

    metrics = calculate_loo_radial_metrics(
        E_utterances[embedding_indices]
    )

    radial_records.append({
        "dataset": dataset,
        "video_id": group["video_id"].iloc[0],
        "conversation_id": conversation_id,
        "level": group["level"].iloc[0],
        "level_label": str(
            group["level_label"].iloc[0]
        ),
        "n_utterances": len(group),
        **metrics,
    })

radial_loo_summary = pd.DataFrame(
    radial_records
)

display(radial_loo_summary.head())

In [ ]:
radial_analysis = radial_loo_summary.merge(
    conversation_geometry[
        [
            "dataset",
            "conversation_id",
            "radial_distance_q90",
        ]
    ],
    on=["dataset", "conversation_id"],
    how="left",
    validate="one_to_one",
)

gap_map = {
    "expert": 0,
    "graduate": 1,
    "undergraduate": 2,
    "teenager": 3,
    "child": 4,
}

radial_analysis["gap_score"] = (
    radial_analysis["level_label"]
    .map(gap_map)
    .astype(float)
)

radial_analysis["gap_c"] = (
    radial_analysis["gap_score"]
    - radial_analysis["gap_score"].mean()
)

radial_analysis["log_n_utterances"] = np.log(
    radial_analysis["n_utterances"]
)

radial_analysis["log_n_c"] = (
    radial_analysis["log_n_utterances"]
    - radial_analysis["log_n_utterances"].mean()
)

radial_analysis["video_group"] = (
    radial_analysis["dataset"].astype(str)
    + "::"
    + radial_analysis["video_id"].astype(str)
)

radial_analysis["level_label"] = pd.Categorical(
    radial_analysis["level_label"],
    categories=[
        "expert",
        "graduate",
        "undergraduate",
        "teenager",
        "child",
    ],
    ordered=True,
)

display(radial_analysis.head())

In [ ]:
import statsmodels.formula.api as smf

OUTCOME = "radial_loo_q90"

model_df = radial_analysis[
    [
        OUTCOME,
        "gap_c",
        "log_n_c",
        "video_group",
        "level_label",
    ]
].dropna().copy()

outcome_mean = model_df[OUTCOME].mean()
outcome_sd = model_df[OUTCOME].std(ddof=1)

model_df["outcome_z"] = (
    model_df[OUTCOME] - outcome_mean
) / outcome_sd

# Unadjusted sensitivity model
radial_model_unadjusted = smf.mixedlm(
    "outcome_z ~ gap_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

# Primary length-adjusted model
radial_model_adjusted = smf.mixedlm(
    "outcome_z ~ gap_c + log_n_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(radial_model_adjusted.summary())

In [ ]:
def extract_gap_result(model, model_name):
    beta = model.fe_params["gap_c"]
    se = model.bse["gap_c"]
    ci = model.conf_int().loc["gap_c"]

    return {
        "model": model_name,

        # Change in outcome SD per one-level increase in gap
        "gap_beta_sd": beta,
        "gap_se": se,
        "ci_lower": ci.iloc[0],
        "ci_upper": ci.iloc[1],
        "p_value": model.pvalues["gap_c"],

        # Child is four gap levels above expert
        "child_expert_difference_sd": 4 * beta,

        # Convert child–expert contrast back to radial-distance units
        "child_expert_difference_raw": (
            4 * beta * outcome_sd
        ),

        "converged": model.converged,
    }


radial_model_results = pd.DataFrame([
    extract_gap_result(
        radial_model_unadjusted,
        "Unadjusted",
    ),
    extract_gap_result(
        radial_model_adjusted,
        "Length-adjusted",
    ),
])

display(radial_model_results)

In [ ]:
radial_video_fe = smf.ols(
    (
        "outcome_z ~ "
        "gap_c + log_n_c + C(video_group)"
    ),
    data=model_df,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": model_df["video_group"],
        "use_correction": True,
    },
)

print(
    radial_video_fe.summary().tables[1]
)

print(
    "\nMixed-model gap coefficient:",
    radial_model_adjusted.params["gap_c"],
)

print(
    "Video-fixed-effect gap coefficient:",
    radial_video_fe.params["gap_c"],
)

In [ ]:
radial_model_results

In [ ]:
import numpy as np
import pandas as pd

def loo_radial_distances(X):
    X = np.asarray(X, dtype=float)

    X_unit = X / np.maximum(
        np.linalg.norm(X, axis=1, keepdims=True),
        1e-12,
    )

    n = len(X_unit)

    if n < 3:
        return np.full(n, np.nan)

    total_vector = X_unit.sum(axis=0)

    # One leave-one-out centroid vector per utterance.
    loo_centroids = total_vector - X_unit

    loo_norms = np.linalg.norm(
        loo_centroids,
        axis=1,
        keepdims=True,
    )

    loo_directions = (
        loo_centroids
        / np.maximum(loo_norms, 1e-12)
    )

    radial_distances = (
        1.0
        - np.sum(
            X_unit * loo_directions,
            axis=1,
        )
    )

    return np.clip(
        radial_distances,
        0.0,
        2.0,
    )


radial_frames = []

for (dataset, conversation_id), group in (
    geometry_utterances.groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    )
):
    group = group.reset_index(drop=True)

    embedding_indices = group[
        "embedding_idx"
    ].to_numpy(dtype=int)

    radial_distance = loo_radial_distances(
        E_utterances[embedding_indices]
    )

    scored = group[
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level",
            "level_label",
            "speaker_role",
            "utterance_id",
            "text",
        ]
    ].copy()

    scored["radial_distance_loo"] = (
        radial_distance
    )

    radial_frames.append(scored)

radial_utterances = pd.concat(
    radial_frames,
    ignore_index=True,
)

In [ ]:
radial_conversation_profiles = (
    radial_utterances
    .groupby(
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level",
            "level_label",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        n_utterances=(
            "radial_distance_loo",
            "size",
        ),
        radial_q10=(
            "radial_distance_loo",
            lambda x: x.quantile(0.10),
        ),
        radial_q25=(
            "radial_distance_loo",
            lambda x: x.quantile(0.25),
        ),
        radial_q50=(
            "radial_distance_loo",
            "median",
        ),
        radial_q75=(
            "radial_distance_loo",
            lambda x: x.quantile(0.75),
        ),
        radial_q90=(
            "radial_distance_loo",
            lambda x: x.quantile(0.90),
        ),
    )
)

radial_conversation_profiles[
    "radial_tail_extension"
] = (
    radial_conversation_profiles["radial_q90"]
    - radial_conversation_profiles["radial_q50"]
)

radial_conversation_profiles[
    "radial_interdecile_range"
] = (
    radial_conversation_profiles["radial_q90"]
    - radial_conversation_profiles["radial_q10"]
)

radial_conversation_profiles[
    "radial_tail_core_ratio"
] = (
    radial_conversation_profiles[
        "radial_tail_extension"
    ]
    / (
        radial_conversation_profiles[
            "radial_q50"
        ]
        + 1e-8
    )
)

display(radial_conversation_profiles.head())

In [ ]:
quantile_grid = np.linspace(
    0.05,
    0.95,
    19,
)

curve_records = []

group_columns = [
    "dataset",
    "video_id",
    "conversation_id",
    "level",
    "level_label",
]

for group_key, group in radial_utterances.groupby(
    group_columns,
    observed=True,
):
    values = (
        group["radial_distance_loo"]
        .dropna()
        .to_numpy()
    )

    quantile_values = np.quantile(
        values,
        quantile_grid,
    )

    metadata = dict(
        zip(group_columns, group_key)
    )

    for probability, value in zip(
        quantile_grid,
        quantile_values,
    ):
        curve_records.append({
            **metadata,
            "quantile": probability,
            "radial_distance": value,
        })

radial_quantile_curves = pd.DataFrame(
    curve_records
)

radial_curve_summary = (
    radial_quantile_curves
    .groupby(
        ["level", "level_label", "quantile"],
        observed=True,
    )["radial_distance"]
    .agg(
        mean="mean",
        sem="sem",
        n_conversations="count",
    )
    .reset_index()
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

LEVEL_ORDER = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

LEVEL_ORDER = [
    level for level in LEVEL_ORDER
    if level in radial_curve_summary[
        "level_label"
    ].astype(str).unique()
]

COLORS = expertise_palette(LEVEL_ORDER)

fig, ax = plt.subplots(figsize=(9, 6))

for level in LEVEL_ORDER:
    subset = (
        radial_curve_summary[
            radial_curve_summary[
                "level_label"
            ].astype(str).eq(level)
        ]
        .sort_values("quantile")
    )

    x = subset["quantile"].to_numpy(float)
    mean = subset["mean"].to_numpy(float)
    sem = subset["sem"].to_numpy(float)

    ax.plot(
        x,
        mean,
        linewidth=2.2,
        color=COLORS[level],
        label=level.title(),
    )

    ax.fill_between(
        x,
        mean - 1.96 * sem,
        mean + 1.96 * sem,
        color=COLORS[level],
        alpha=0.12,
    )

for quantile in [0.10, 0.50, 0.90]:
    ax.axvline(
        quantile,
        color="gray",
        linestyle=":",
        linewidth=1,
        alpha=0.6,
    )

ax.set_xlabel("Utterance radial-distance quantile")
ax.set_ylabel(
    "Mean leave-one-out radial distance"
)
ax.set_title(
    "Conversation-centered semantic radial profiles"
)
ax.legend(
    title="Partner expertise",
    frameon=False,
)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(
    data=radial_conversation_profiles,
    x="radial_q50",
    y="radial_tail_extension",
    hue="level_label",
    hue_order=LEVEL_ORDER,
    palette=COLORS,
    s=65,
    alpha=0.65,
    ax=ax,
)

level_means = (
    radial_conversation_profiles
    .groupby(
        "level_label",
        observed=True,
    )[
        [
            "radial_q50",
            "radial_tail_extension",
        ]
    ]
    .mean()
    .reset_index()
)

sns.scatterplot(
    data=level_means,
    x="radial_q50",
    y="radial_tail_extension",
    hue="level_label",
    hue_order=LEVEL_ORDER,
    palette=COLORS,
    marker="D",
    s=180,
    edgecolor="black",
    linewidth=1,
    legend=False,
    ax=ax,
)

ax.set_xlabel(
    "Core radius: median radial distance"
)
ax.set_ylabel(
    "Tail extension: q90 − median"
)
ax.set_title(
    "Semantic contraction and peripheral extension"
)
ax.legend(
    title="Partner expertise",
    frameon=False,
)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()

In [2]:
import plotly.express as px

def plot_metric_distribution_interactive(
    df,
    metric,
    ylabel=None
):
    fig = px.box(
        df,
        x="level_label",
        y=metric,
        category_orders={"level_label": level_order},
        points="all",
        hover_data={
            "video_id": True,
            "conversation_id": True,
            "level_label": True,
            metric: ":.4f",
            "n_utterances": True,
            "n_words": True,
        },
        labels={
            "level_label": "Partner expertise → smaller expertise gap",
            metric: ylabel or metric,
            "video_id": "Topic / Video",
            "conversation_id": "Conversation",
            "n_utterances": "Utterances",
            "n_words": "Words",
        },
        title=ylabel or metric,
    )

    fig.update_traces(
        jitter=0.22,
        marker=dict(
            size=7,
            opacity=0.7
        )
    )

    fig.update_layout(
        width=850,
        height=500,
        xaxis_title="Partner expertise → smaller expertise gap",
        yaxis_title=ylabel or metric,
        template="simple_white"
    )

    fig.show()

In [26]:
plot_metric_distribution_interactive(conversation_geometry, 'participation_ratio')

In [38]:
conversation_geometry.columns

Index(['dataset', 'conversation_id', 'video_id', 'file_number', 'level',
       'level_label', 'n_utterances', 'n_pairs', 'n_words', 'n_turns',
       'embedding_dimension', 'k_used', 'pairwise_cosine_similarity_mean',
       'pairwise_cosine_similarity_sd', 'pairwise_cosine_similarity_q10',
       'pairwise_cosine_similarity_median', 'pairwise_cosine_similarity_q90',
       'centroid_resultant_length', 'centroid_dispersion_cosine',
       'radial_distance_mean_check', 'radial_distance_median',
       'radial_distance_q90', 'pairwise_distance_mean', 'pairwise_distance_sd',
       'pairwise_distance_q10', 'pairwise_distance_median',
       'pairwise_distance_q90', 'pairwise_semantic_span',
       'nearest_neighbor_distance_mean', 'nearest_neighbor_distance_median',
       'knn_distance_median', 'maximum_available_rank', 'numerical_rank',
       'total_variance', 'spectral_entropy', 'spectral_entropy_normalized',
       'spectral_effective_rank', 'participation_ratio', 'pc1_variance_shar